# UCS3564 – Natural Language Processing
## A3: Part-of-Speech Tagging using a Hidden Markov Model
Transition / emission estimation with add-one smoothing + Viterbi decoding on the NLTK Penn Treebank sample.

In [ ]:
# Setup: run once
!pip install -q nltk numpy pandas
import nltk
nltk.download("treebank", quiet=True)

In [ ]:
import random
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from nltk.corpus import treebank

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 15)

START, UNK = "<START>", "<UNK>"

## Data: 80 / 20 split

In [ ]:
tagged_sents = list(treebank.tagged_sents())
random.seed(42)
random.shuffle(tagged_sents)

split_idx = int(0.8 * len(tagged_sents))
train_sents, test_sents = tagged_sents[:split_idx], tagged_sents[split_idx:]
print(f"Total sentences: {len(tagged_sents)} | Train: {len(train_sents)} | Test: {len(test_sents)}")

## Task 1 – Vocabulary and tagset (10 marks)

In [ ]:
def build_vocab_and_tagset(sentences):
    """Return sorted vocabulary (with <UNK>) and sorted tagset from training sentences."""
    vocab = {word.lower() for sent in sentences for word, _ in sent}
    tagset = {tag for sent in sentences for _, tag in sent}
    return sorted(vocab) + [UNK], sorted(tagset)


vocab, tagset = build_vocab_and_tagset(train_sents)
word_to_idx = {w: i for i, w in enumerate(vocab)}
tag_to_idx = {t: i for i, t in enumerate(tagset)}

print("Vocabulary size (incl. <UNK>):", len(vocab))
print("Tagset size:", len(tagset))
print("Tags:", tagset)

## Task 2 – Transition matrix P(tᵢ | tᵢ₋₁) with add-one smoothing (20 marks)

`<START>` is added as an extra previous state so that the first tag of each sentence is also modelled:

P(tᵢ | tᵢ₋₁) = (C(tᵢ₋₁, tᵢ) + 1) / (C(tᵢ₋₁) + T), where T = number of tags.

In [ ]:
def estimate_transition_matrix(sentences, tagset, tag_to_idx):
    """Return (T+1) x T matrix; row 0 is the <START> state."""
    num_tags = len(tagset)
    counts = np.zeros((num_tags + 1, num_tags))
    for sent in sentences:
        prev = 0                                   # row 0 = <START>
        for _, tag in sent:
            counts[prev, tag_to_idx[tag]] += 1
            prev = tag_to_idx[tag] + 1             # rows 1..T = tags
    smoothed = counts + 1                          # add-one
    return smoothed / smoothed.sum(axis=1, keepdims=True)


transition_prob = estimate_transition_matrix(train_sents, tagset, tag_to_idx)
transition_df = pd.DataFrame(transition_prob, index=[START] + tagset, columns=tagset)

print("Transition matrix shape (rows = previous tag, cols = current tag):", transition_df.shape)
print("Rows sum to 1:", np.allclose(transition_prob.sum(axis=1), 1))
transition_df.round(4)

## Task 3 – Emission matrix P(wᵢ | tᵢ) with add-one smoothing (20 marks)

P(w | t) = (C(t, w) + 1) / (C(t) + V), where V = vocabulary size (including `<UNK>`).

In [ ]:
def estimate_emission_matrix(sentences, tagset, vocab, tag_to_idx, word_to_idx):
    counts = np.zeros((len(tagset), len(vocab)))
    for sent in sentences:
        for word, tag in sent:
            counts[tag_to_idx[tag], word_to_idx[word.lower()]] += 1
    smoothed = counts + 1                          # add-one (also gives <UNK> mass)
    return smoothed / smoothed.sum(axis=1, keepdims=True)


emission_prob = estimate_emission_matrix(train_sents, tagset, vocab, tag_to_idx, word_to_idx)
emission_df = pd.DataFrame(emission_prob, index=tagset, columns=vocab)
print("Emission matrix shape (tags x words):", emission_df.shape)

sample_words = ["the", "market", "said", "of", "year"]
sample_words = [w for w in sample_words if w in word_to_idx]
print("\nP(word | tag) for words:", sample_words)
emission_df[sample_words].round(6)

## Task 4 – Viterbi decoding with backpointer table (10 marks)

In [ ]:
def viterbi(words, transition_prob, emission_prob, tagset, word_to_idx):
    """Return best tag sequence, Viterbi (log-prob) table and backpointer table."""
    num_tags, n = len(tagset), len(words)
    log_trans, log_emit = np.log(transition_prob), np.log(emission_prob)
    obs = [word_to_idx.get(w.lower(), word_to_idx[UNK]) for w in words]

    score = np.full((num_tags, n), -np.inf)
    backptr = np.zeros((num_tags, n), dtype=int)

    score[:, 0] = log_trans[0, :] + log_emit[:, obs[0]]          # from <START>
    for t in range(1, n):
        # candidate[prev, cur] = score[prev, t-1] + log P(cur | prev)
        candidate = score[:, t - 1][:, None] + log_trans[1:, :]
        backptr[:, t] = np.argmax(candidate, axis=0)
        score[:, t] = np.max(candidate, axis=0) + log_emit[:, obs[t]]

    best_last = int(np.argmax(score[:, -1]))
    path = [best_last]
    for t in range(n - 1, 0, -1):
        path.append(int(backptr[path[-1], t]))
    path.reverse()
    return [tagset[i] for i in path], score, backptr


test_sent = test_sents[0]
test_words = [w for w, _ in test_sent]
true_tags = [t for _, t in test_sent]
pred_tags, score, backptr = viterbi(test_words, transition_prob, emission_prob, tagset, word_to_idx)

print("Sentence:", " ".join(test_words))
print(f"\n{'Word':<15}{'Gold':<8}{'Predicted':<10}")
for w, g, p in zip(test_words, true_tags, pred_tags):
    print(f"{w:<15}{g:<8}{p:<10}{'' if g == p else '  <-- error'}")

# Backpointer table: entry = best previous tag for (current tag, position)
backptr_df = pd.DataFrame([[tagset[backptr[i, j]] if j > 0 else START for j in range(len(test_words))]
                           for i in range(len(tagset))], index=tagset, columns=test_words)
print("\nBackpointer table (rows = current tag, cols = word; value = best previous tag):")
backptr_df

In [ ]:
# Viterbi log-probability table for the same sentence
pd.DataFrame(score, index=tagset, columns=test_words).round(2)

## Task 5 – Evaluation: tag-wise and overall accuracy (10 marks)

In [ ]:
def evaluate(test_sentences):
    correct, total = Counter(), Counter()
    for sent in test_sentences:
        words = [w for w, _ in sent]
        gold = [t for _, t in sent]
        pred, _, _ = viterbi(words, transition_prob, emission_prob, tagset, word_to_idx)
        for g, p in zip(gold, pred):
            total[g] += 1
            correct[g] += int(g == p)
    return correct, total


correct, total = evaluate(test_sents)

rows = [(tag, total[tag], correct[tag], correct[tag] / total[tag]) for tag in sorted(total)]
accuracy_df = pd.DataFrame(rows, columns=["Tag", "Count", "Correct", "Accuracy"])
print(accuracy_df.to_string(index=False, formatters={"Accuracy": "{:.4f}".format}))

overall = sum(correct.values()) / sum(total.values())
print(f"\nOverall accuracy: {overall:.4f}  ({sum(correct.values())}/{sum(total.values())})")

### Observations
* Add-one smoothing gives unseen words a small, equal emission probability under every tag, so the
  decision for `<UNK>` words is dominated by the transition probabilities.
* Frequent closed-class tags (DT, IN, ',', '.') reach very high accuracy; rare tags and open-class tags
  with many unseen words (NNP, JJ, FW) are weaker.